In [14]:
import sys

from copy import deepcopy
from lightning import pytorch as pl
from pathlib import Path

import pandas as pd
import numpy as np
import torch

from dataclasses import InitVar, dataclass
from typing import Sequence
from rdkit import Chem
from rdkit.Chem import Mol, Draw
from rdkit.Chem.rdchem import Atom, Bond, BondType

from chemprop.featurizers.atom import MultiHotAtomFeaturizer
from chemprop.featurizers.bond import MultiHotBondFeaturizer
from chemprop.featurizers.molgraph.molecule import SimpleMoleculeMolGraphFeaturizer

from chemprop.data.molgraph import MolGraph
from chemprop.featurizers.base import GraphFeaturizer
from chemprop.featurizers.molgraph.mixins import _MolGraphFeaturizerMixin
from typing import Any
from chemprop import data, featurizers, models

import shap
from rdkit.Chem.rdchem import HybridizationType

In [ ]:
def get_predictions():
    pass

class MoleculeModelWrapper:
    pass

def binary_masker():
    pass

keep_features: list[int]
feature_choice = np.array([keep_features])
model_wrapper = MoleculeModelWrapper()

explainer = shap.PermutationExplainer(model_wrapper, masker=binary_masker)
explanation = explainer(feature_choice, max_evals=100)

In [6]:
# Example usage
atomic_nums = [6, 7, 8]
degrees = [1, 2, 3]
formal_charges = []
chiral_tags = [0, 1, 2]
num_Hs = [0, 1, 2]
hybridizations = [1, 2, 3]

keep_features_all = [True] * 8
keep_features_some = [True, True, False, True, False, True, True, False]
keep_features_none = [False] * 8

featurizer_all = CustomMultiHotAtomFeaturizer(
    atomic_nums=atomic_nums,
    degrees=degrees,
    formal_charges=formal_charges,
    chiral_tags=chiral_tags,
    num_Hs=num_Hs,
    hybridizations=hybridizations,
    keep_features=keep_features_all
)

print(featurizer_all._subfeats)

featurizer_some = CustomMultiHotAtomFeaturizer(
    atomic_nums=atomic_nums,
    degrees=degrees,
    formal_charges=formal_charges,
    chiral_tags=chiral_tags,
    num_Hs=num_Hs,
    hybridizations=hybridizations,
    keep_features=keep_features_some
)

featurizer_none = CustomMultiHotAtomFeaturizer(
    atomic_nums=atomic_nums,
    degrees=degrees,
    formal_charges=formal_charges,
    chiral_tags=chiral_tags,
    num_Hs=num_Hs,
    hybridizations=hybridizations,
    keep_features=keep_features_none
)

mol = Chem.MolFromSmiles('CCO')
atom = mol.GetAtomWithIdx(0)  # Get the first atom

features = featurizer_all(atom)
print("Atom features all:", features)

features = featurizer_some(atom)
print("Atom features some:", features)

features = featurizer_none(atom)
print("Atom features none:", features)

[{6: 0, 7: 1, 8: 2}, {1: 1, 2: 2, 3: 3}, {}, {0: 0, 1: 1, 2: 2}, {0: 0, 1: 1, 2: 2}, {1: 0, 2: 1, 3: 2}]
subfeat: [{6: 0, 7: 1, 8: 2}, {1: 1, 2: 2, 3: 3}, {}, {0: 0, 1: 1, 2: 2}, {0: 0, 1: 1, 2: 2}, {1: 0, 2: 1, 3: 2}]
Atom features all: [1.      0.      0.      0.      0.      0.      0.      1.      1.
 1.      0.      0.      0.      0.      0.      0.      1.      0.
 0.      0.      1.      0.      0.12011]
subfeat: [{6: 0, 7: 1, 8: 2}, {1: 1, 2: 2, 3: 3}, {}, {0: 0, 1: 1, 2: 2}, {0: 0, 1: 1, 2: 2}, {1: 0, 2: 1, 3: 2}]
Atom features some: [1. 0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
subfeat: [{6: 0, 7: 1, 8: 2}, {1: 1, 2: 2, 3: 3}, {}, {0: 0, 1: 1, 2: 2}, {0: 0, 1: 1, 2: 2}, {1: 0, 2: 1, 3: 2}]
Atom features none: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [20]:
bond_types = [BondType.SINGLE, BondType.DOUBLE, BondType.TRIPLE, BondType.AROMATIC]
stereos = [0, 1, 2, 3, 4, 5]
keep_features_all = [True] * 4
keep_features_some = [True, False, True, False]
keep_features_none = [False] * 4

featurizer_all = CustomMultiHotBondFeaturizer(
    bond_types=bond_types,
    stereos=stereos,
    keep_features=keep_features_all
)

featurizer_some = CustomMultiHotBondFeaturizer(
    bond_types=bond_types,
    stereos=stereos,
    keep_features=keep_features_some
)

featurizer_none = CustomMultiHotBondFeaturizer(
    bond_types=bond_types,
    stereos=stereos,
    keep_features=keep_features_none
)

mol = Chem.MolFromSmiles('CCO')
bond = mol.GetBondWithIdx(0)  # Get the first bond

features = featurizer_all(bond)
print("Bond features all:", features)

features = featurizer_some(bond)
print("Bond features some:", features)

features = featurizer_none(bond)
print("Bond features none:", features)

Bond features all: [0 1 0 0 0 0 0 1 0 0 0 0 0 0]
Bond features some: [0 1 0 0 0 0 0 0 0 0 0 0 0 0]
Bond features none: [0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [ ]:
atom_featurizer = CustomMultiHotAtomFeaturizer.v2()
bond_featurizer = CustomMultiHotBondFeaturizer()

def get_predictions(keep_atom_features: list[bool] | None, keep_bond_features: list[bool] | None, mol: str) -> float:
    featurizer = CustomSimpleMoleculeMolGraphFeaturizer(
        atom_featurizer=atom_featurizer,
        bond_featurizer=bond_featurizer,
        keep_atom_features=keep_atom_features,
        keep_bond_features=keep_bond_features
    )
    test_data = [data.MoleculeDatapoint.from_smi(mol)]
    test_dset = data.MoleculeDataset(test_data, featurizer=featurizer)
    test_loader = data.build_dataloader(test_dset, shuffle=False, batch_size=1)

    with torch.inference_mode():
        trainer = pl.Trainer(
            logger=False,
            enable_progress_bar=False,
            accelerator="cpu",
            devices=1
        )
        test_preds = trainer.predict(mpnn, test_loader)
    return test_preds[0][0]

In [ ]:
class MoleculeModelWrapper:
    def __init__(self, mol: str):
        self.mol = mol

    def __call__(self, X) -> np.ndarray:
        preds = []
        for keep_features in X:
            try:
                # unpacking X, indices corresponds to feature orders from default chemprop featurizer, adapt as needed
                keep_atom_features = keep_features[:8] # 8 atom features
                keep_bond_features = keep_features[8:] # 4 bond features
            except:
                print(f"Invalid input: {keep_features}")
                raise
            pred = get_predictions(keep_atom_features, keep_bond_features, self.mol)
            preds.append([pred.item()])
        return np.array(preds)

In [ ]:
def binary_masker(binary_mask, x):
    masked_x = deepcopy(x)
    masked_x[binary_mask == 0] = 0
    return np.array([masked_x])

In [8]:
np.array([[1] * 12]).shape

(1, 12)

In [ ]:
def get_predictions(
    mask: np.ndarray,
    datapoint: MoleculeDatapoint,
    mask_config: ShapMaskConfig,
    base_atom_featurizer: MultiHotAtomFeaturizer,
    base_bond_featurizer: MultiHotBondFeaturizer,
    mpnn: models.MPNN,
    trainer: pl.Trainer,
) -> float:
    """Get a model prediction for one molecule with a feature group mask applied.
 
    Parameters
    ----------
    mask:
        Flat binary mask vector of length mask_config.total.
    datapoint:
        The MoleculeDatapoint for the molecule (with precomputed V_f, E_f, x_d).
    mask_config:
        Defines the structure of the mask vector.
    base_atom_featurizer:
        The atom featurizer used during training (to read its config).
    base_bond_featurizer:
        The bond featurizer used during training.
    mpnn:
        The trained Chemprop MPNN model.
    trainer:
        Lightning Trainer configured for inference.
    """
    unpacked = mask_config.unpack(mask)
 
    # build custom featurizers with the default feature groups masked
    atom_featurizer = CustomMultiHotAtomFeaturizer.from_base(
        base_atom_featurizer,
        keep_features=unpacked["keep_atom_features"],
    )
    bond_featurizer = CustomMultiHotBondFeaturizer(
        bond_types=base_bond_featurizer.bond_types,
        stereos=base_bond_featurizer.stereos,
        keep_features=unpacked["keep_bond_features"],
    )
 
    # apply mask to extra features
    V_f = mask_extra_features(datapoint.V_f, unpacked["keep_extra_atom_groups"])
    E_f = mask_extra_features(datapoint.E_f, unpacked["keep_extra_bond_groups"])
    x_d = mask_mol_features(datapoint.x_d, unpacked["keep_mol_features"])
 
    # build a fresh datapoint with masked extra features but same SMILES
    masked_datapoint = MoleculeDatapoint.from_smi(
        smi=datapoint.name,
        V_f=V_f,
        E_f=E_f,
        x_d=x_d,
        keep_h=True,
    )
 
    # build dataset and dataloader with masked featurizer
    extra_atom_fdim = datapoint.V_f.shape[1] if datapoint.V_f is not None else 0
    extra_bond_fdim = datapoint.E_f.shape[1] if datapoint.E_f is not None else 0
 
    featurizer = SimpleMoleculeMolGraphFeaturizer(
        atom_featurizer=atom_featurizer,
        bond_featurizer=bond_featurizer,
        extra_atom_fdim=extra_atom_fdim,
        extra_bond_fdim=extra_bond_fdim,
    )
 
    dataset = MoleculeDataset([masked_datapoint], featurizer=featurizer)
    loader = data.build_dataloader(dataset, shuffle=False, batch_size=1)
 
    preds = trainer.predict(mpnn, loader)
    # preds is a list of tensors, one per batch
    return preds[0][0].item()

In [31]:
datapoint = data.MoleculeDatapoint.from_smi('[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H:19])[c:6]([C:7]([H:20])([H:21])[H:22])[c:8]1[O:9][C:10]([C:11]([O:12][H:25])=[O:13])([H:23])[H:24])([H:14])([H:15])[H:16]', keep_h=True)

In [36]:
datapoint

MoleculeDatapoint(mol=<rdkit.Chem.rdchem.Mol object at 0x000002495EF92A40>, y=None, weight=1.0, gt_mask=None, lt_mask=None, x_d=None, x_phase=None, name='[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H:19])[c:6]([C:7]([H:20])([H:21])[H:22])[c:8]1[O:9][C:10]([C:11]([O:12][H:25])=[O:13])([H:23])[H:24])([H:14])([H:15])[H:16]', V_f=None, E_f=None, V_d=None)